# fluxcharge tutorial

From a lumped-element circuit to its **analytical Hamiltonian**, then a
**numerical spectrum in GHz** — including the gyrator and quantum-phase-slip
circuits that are unique to the flux–charge symmetric formalism.

Run top to bottom. Needs `numpy` (`pip install "fluxcharge[numeric]"`).


In [ ]:
import numpy as np, math
import matplotlib.pyplot as plt
import fluxcharge as fc
from fluxcharge import library


## 1. Transmon — symbolic H, then GHz spectrum
Build it from the library, look at the exact Hamiltonian, then diagonalize
in physical units.


In [ ]:
tr = library.transmon()                 # JJ shunted by a capacitor
res = tr.hamiltonian(ground='v1')
res.H                                    # symbolic Hamiltonian


In [ ]:
res.report()                             # reduction + commutators + mode info
print(res.report())


In [ ]:
p = tr.natural_params({'C': '70fF', 'E_J': '15GHz'})  # -> eigenvalues in GHz
ev = res.eigenenergies(p, n_levels=5, cutoffs={'q_f1': 81})
print('levels (GHz):', np.round(ev, 4))
print('f01 = %.4f GHz   anharmonicity = %.4f GHz'
      % (ev[1]-ev[0], (ev[2]-ev[1])-(ev[1]-ev[0])))


In [ ]:
# charge dispersion: sweep the gate (offset) charge
cpb = library.cooper_pair_box()
rc = cpb.hamiltonian(ground='v1')
ng = np.linspace(-1, 1, 81)
rc.plot_spectrum('n_g_v2', ng, cpb.natural_params({'C':'90fF','E_J':'8GHz'}),
                 n_levels=3, cutoffs={'q_f1':81}, relative=True)
plt.show()


## 2. Fluxonium — external flux and the sweet spot
An external flux threads the JJ–inductor loop; the symbol equals the
physical flux (period $2\pi$, sweet spot at $\pi$).


In [ ]:
fx = library.fluxonium()
rf = fx.hamiltonian(ground='v1', open_loops='f3')
pf = fx.natural_params({'E_J':'5GHz', 'C':'1GHz', 'L':'1GHz'})  # C,L as E_C,E_L
phi = np.linspace(0, 2*np.pi, 81)
rf.plot_spectrum('phi_ext_f1', phi, pf, n_levels=4, cutoffs={'phi_v2':60}, relative=True)
plt.show()


In [ ]:
# potential + wavefunctions at the half-flux sweet spot
pf_sweet = dict(pf); pf_sweet['phi_ext_f1'] = math.pi
rf.plot_potential_wavefunctions(pf_sweet, n_levels=4, cutoffs={'phi_v2':60})
plt.show()


In [ ]:
# first-order flux sensitivity is zero at the sweet spot
df, d2f = rf.transition_sensitivity('phi_ext_f1', pf_sweet, cutoffs={'phi_v2':60})
print('df01/dphi = %.4f GHz/rad,  d2f01/dphi2 = %.4f' % (df, d2f))


## 3. The gyrator circulator (non-reciprocal)
The manuscript's worked example — nothing reciprocal can represent it.
The gyrator gives a `phi*q` cross term, so there is no scalar potential and
we show the level diagram.


In [ ]:
cir = library.circulator()
rcir = cir.hamiltonian(ground='v1', open_loops='f4', canonical=True)
rcir.H


In [ ]:
ev = rcir.eigenenergies({'E_J':10.0,'C':1.0,'G':0.5}, n_levels=6, cutoffs={'phi_v2':60})
rcir.plot_energy_levels({'E_J':10.0,'C':1.0,'G':0.5}, n_levels=6, cutoffs={'phi_v2':60})
plt.show()


## 4. Quantum phase-slip qubit — the LCG dual
A coherent phase slip shunting an inductor is the charge-space dual of the
transmon: it quantizes the **flux** and shares the transmon's spectrum under
$E_S\leftrightarrow E_J,\ L\leftrightarrow C$.


In [ ]:
qp = library.phase_slip_qubit(charge_bias=False)
rq = qp.hamiltonian(ground='v1')
ev_q = rq.eigenenergies({'E_S':10.0,'L':1.0}, n_levels=5, cutoffs={'phi_v2':80})
ev_t = library.transmon().hamiltonian(ground='v1').eigenenergies(
    {'E_J':10.0,'C':1.0}, n_levels=5, cutoffs={'q_f1':81})
print('phase-slip:', np.round(ev_q,5))
print('transmon  :', np.round(ev_t,5))
print('max |diff| =', np.max(np.abs(ev_q-ev_t)))


In [ ]:
# dual() also maps the circuit explicitly (loop<->node, C<->L, JJ<->QPS)
from fluxcharge import dual, to_netlist
print(to_netlist(dual(library.transmon())))


## 5. Matrix elements and a T1 estimate
Matrix elements are exact; T1 follows Fermi's golden rule given a spectral
density $S(\omega)$ (the physics is in $S$ — supply your own).


In [ ]:
M = res.matrix_elements('q_f1', p, n_levels=4, cutoffs={'q_f1':81})
print('|<i|n|j>| =\n', np.round(np.abs(M), 3))
# golden rule with a placeholder flat spectral density
T1, rate = res.t1(p, 'q_f1', lambda w: 1.0, cutoffs={'q_f1':81})
print('relaxation rate (flat S) =', round(rate, 4))
